# Cut-in Risk Analysis: Thesis Runner

This notebook runs the thesis pipeline in a simple and reproducible way.

## How To Use

1. Run setup and path check cells.
2. Choose one mode: `quick`, `full`, or `custom`.
3. Run the execution cell.
4. Check outputs in the final cells.

In [14]:
from __future__ import annotations

import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

Project root: /Users/sandeep/IdeaProjects/cutin-risk-analysis


## Path Check

In [15]:
from cutin_risk.paths import dataset_root_path, outputs_root_path, step14_codes_csv_path

dataset_root = dataset_root_path()
outputs_root = outputs_root_path()
codes_csv = step14_codes_csv_path()

print('dataset_root:', dataset_root)
print('outputs_root:', outputs_root)
print('step14_codes_csv:', codes_csv)

check = pd.DataFrame([
    {'item': 'dataset_root', 'path': str(dataset_root), 'exists': dataset_root.exists()},
    {'item': 'outputs_root', 'path': str(outputs_root), 'exists': outputs_root.exists()},
    {'item': 'step14_codes_csv', 'path': str(codes_csv), 'exists': codes_csv.exists()},
])
display(check)

dataset_root: /Users/sandeep/IdeaProjects/cutin-risk-analysis/data/raw/highD-dataset-v1.0/data
outputs_root: /Users/sandeep/IdeaProjects/cutin-risk-analysis/outputs
step14_codes_csv: /Users/sandeep/IdeaProjects/cutin-risk-analysis/outputs/reports/step14_sfc_binary/sfc_binary_codes_long_hilbert.csv


,item,path,exists
0,dataset_root,/Users/sandeep/IdeaProjects/cutin-risk-analysi...,True
1,outputs_root,/Users/sandeep/IdeaProjects/cutin-risk-analysi...,True
2,step14_codes_csv,/Users/sandeep/IdeaProjects/cutin-risk-analysi...,True


## Run Mode

In [18]:
# Choose one: 'quick', 'full', 'custom'
MODE = 'full'

# Only used when MODE == 'custom'
CUSTOM_STEPS = ['step08', 'step09', 'step10']

# Step09 option
RECORDINGS_FOR_BATCH = '1-10'

# Safety switch (recommended): keep False until paths/settings are confirmed
RUN_PIPELINE = True

DRY_RUN = False
STOP_ON_ERROR = True

print('MODE:', MODE)
print('CUSTOM_STEPS:', CUSTOM_STEPS)
print('RECORDINGS_FOR_BATCH:', RECORDINGS_FOR_BATCH)
print('RUN_PIPELINE:', RUN_PIPELINE)
print('DRY_RUN:', DRY_RUN)
print('STOP_ON_ERROR:', STOP_ON_ERROR)

MODE: full
CUSTOM_STEPS: ['step08', 'step09', 'step10']
RECORDINGS_FOR_BATCH: 1-10
RUN_PIPELINE: True
DRY_RUN: False
STOP_ON_ERROR: True


## Step List

In [10]:
STEP_COMMANDS = {
    'step01': [sys.executable, 'scripts/step01_recording_report.py'],
    'step02': [sys.executable, 'scripts/step02_lane_change_report.py'],
    'step03': [sys.executable, 'scripts/step03_cutin_report.py'],
    'step04': [sys.executable, 'scripts/step04_risk_metrics_report.py'],
    'step05': [sys.executable, 'scripts/step05_visualize_top_cutins.py'],
    'step06': [sys.executable, 'scripts/step06_neighbor_reconstruction_report.py'],
    'step07': [sys.executable, 'scripts/step07_xy_lane_pipeline_report.py'],
    'step08': [sys.executable, 'scripts/step08_stage_features.py', '--recording-id', '01'],
    'step09': [sys.executable, 'scripts/step09_batch_stage_features.py', '--recordings', RECORDINGS_FOR_BATCH],
    'step10': [sys.executable, 'scripts/step10_risk_report.py'],
    'step11': [sys.executable, 'scripts/step11_early_warning_rule_search.py'],
    'step12': [sys.executable, 'scripts/step12_early_warning_logreg.py'],
    'step13a': [sys.executable, 'scripts/step13a_logreg_tuned.py'],
    'step13b': [sys.executable, 'scripts/step13b_realistic_follower_warning.py'],
    'step14': [sys.executable, 'scripts/step14_sfc_binary_encode.py'],
    'step14r': [sys.executable, 'scripts/step14_sfc_binary_report.py'],
    'step15a': [sys.executable, 'scripts/step15a_sfc_mirror_normalize.py'],
    'step15b': [sys.executable, 'scripts/step15b_sfc_weighted_stage_features.py'],
    'step15c': [sys.executable, 'scripts/step15c_sfc_prediction.py'],
    'step16': [sys.executable, 'scripts/step16_sfc_predict_lanechange_cutin.py'],
}

QUICK_STEPS = ['step02', 'step03', 'step04', 'step08']
FULL_STEPS = list(STEP_COMMANDS.keys())

display(pd.DataFrame({'available_steps': list(STEP_COMMANDS.keys())}))

,available_steps
0,step01
1,step02
2,step03
3,step04
4,step05
5,step06
6,step07
7,step08
8,step09
9,step10


## Run Pipeline

In [19]:
def run_steps(step_ids: list[str]) -> None:
    for sid in step_ids:
        if sid not in STEP_COMMANDS:
            raise ValueError(f'Unknown step: {sid}')
        cmd = STEP_COMMANDS[sid]
        print('\n' + '=' * 20 + f' {sid} ' + '=' * 20)
        print('$', ' '.join(shlex.quote(x) for x in cmd))

        if DRY_RUN:
            print('[DRY_RUN] skipped')
            continue

        rc = subprocess.call(cmd, cwd=str(PROJECT_ROOT))
        if rc != 0:
            print(f'[ERROR] {sid} failed with exit code {rc}')
            if STOP_ON_ERROR:
                raise RuntimeError(f'Stopped at {sid}')


if MODE == 'quick':
    selected = QUICK_STEPS
elif MODE == 'full':
    selected = FULL_STEPS
elif MODE == 'custom':
    selected = CUSTOM_STEPS
else:
    raise ValueError("MODE must be 'quick', 'full', or 'custom'")

print('Selected steps:', selected)
if RUN_PIPELINE:
    run_steps(selected)
else:
    print('RUN_PIPELINE is False. Set RUN_PIPELINE = True to execute selected steps.')

Selected steps: ['step01', 'step02', 'step03', 'step04', 'step05', 'step06', 'step07', 'step08', 'step09', 'step10', 'step11', 'step12', 'step13a', 'step13b', 'step14', 'step14r', 'step15a', 'step15b', 'step15c', 'step16']

==================== step01 ====================
$ /Users/sandeep/IdeaProjects/cutin-risk-analysis/.venv/bin/python scripts/step01_recording_report.py


--------------------------------------------------------------------------------

Step 01: Report for recording 01
Rows: 348750
Vehicles: 1047
Time range: 0.04s .. 901.56s
Lane IDs: (2, 3, 5, 6)
Optional columns present: ('backSightDistance', 'dhw', 'frontSightDistance', 'precedingXVelocity', 'thw', 'ttc')
Duplicate (id, frame) rows: 0
Vehicles with non-monotonic time: 0
Neighbor id integrity: 24000/24000 OK
Neighbor id integrity failures: 0

--------------------------------------------------------------------------------

Step 01: Report for recording 02
Rows: 378115
Vehicles: 1113
Time range: 0.04s .. 1003.04s
Lane IDs: (2, 3, 5, 6)
Optional columns present: ('backSightDistance', 'dhw', 'frontSightDistance', 'precedingXVelocity', 'thw', 'ttc')
Duplicate (id, frame) rows: 0
Vehicles with non-monotonic time: 0
Neighbor id integrity: 24000/24000 OK
Neighbor id integrity failures: 0

--------------------------------------------------------------------------------

Step 01: Report for rec


==================== step02 ====================
$ /Users/sandeep/IdeaProjects/cutin-risk-analysis/.venv/bin/python scripts/step02_lane_change_report.py
Step 02: Lane Change Report
Recording: 01
Vehicles: 1047
Lane change events: 118

Top transitions:
  3 -> 2: 35
  2 -> 3: 33
  5 -> 6: 28
  6 -> 5: 22

All lane change events:
LaneChangeEvent(vehicle_id=9, from_lane=2, to_lane=3, start_frame=110, end_frame=223, start_time=4.4, end_time=8.92)
LaneChangeEvent(vehicle_id=11, from_lane=6, to_lane=5, start_frame=164, end_frame=235, start_time=6.56, end_time=9.4)
LaneChangeEvent(vehicle_id=23, from_lane=3, to_lane=2, start_frame=628, end_frame=653, start_time=25.12, end_time=26.12)
LaneChangeEvent(vehicle_id=30, from_lane=2, to_lane=3, start_frame=588, end_frame=785, start_time=23.52, end_time=31.4)
LaneChangeEvent(vehicle_id=48, from_lane=3, to_lane=2, start_frame=1148, end_frame=1268, start_time=45.92, end_time=50.72)
LaneChangeEvent(vehicle_id=58, from_lane=2, to_lane=3, start_frame=1368

usage: step15a_sfc_mirror_normalize.py [-h] --codes-csv CODES_CSV
                                       [--events-csv EVENTS_CSV]
                                       [--from-col FROM_COL] [--to-col TO_COL]
                                       [--canonical-direction {positive,negative}]
                                       [--out-dir OUT_DIR]
step15a_sfc_mirror_normalize.py: error: the following arguments are required: --codes-csv


RuntimeError: Stopped at step15a

## Output Check

In [12]:
from cutin_risk.paths import outputs_root_path

out = outputs_root_path()
targets = {
    'step8_dir': out / 'reports/step8_recording_01',
    'step9_merged': out / 'reports/step9_batch/cutin_stage_features_merged.csv',
    'step10_dir': out / 'reports/step10_risk',
    'step14_codes': out / 'reports/step14_sfc_binary/sfc_binary_codes_long_hilbert.csv',
}

rows = [{'name': k, 'path': str(v), 'exists': v.exists()} for k, v in targets.items()]
display(pd.DataFrame(rows))

merged = targets['step9_merged']
if merged.exists():
    df = pd.read_csv(merged)
    print('Merged rows:', len(df), 'columns:', len(df.columns))
    display(df.head(5))

,name,path,exists
0,step8_dir,/Users/sandeep/IdeaProjects/cutin-risk-analysi...,True
1,step9_merged,/Users/sandeep/IdeaProjects/cutin-risk-analysi...,True
2,step10_dir,/Users/sandeep/IdeaProjects/cutin-risk-analysi...,True
3,step14_codes,/Users/sandeep/IdeaProjects/cutin-risk-analysi...,True


Merged rows: 1282 columns: 50


,recording_id,cutter_id,follower_id,from_lane,to_lane,t0_frame,t0_time,dhw_min_total,thw_min_total,ttc_min_total,...,recovery_rows,recovery_dhw_min,recovery_thw_min,recovery_ttc_min,recovery_cutter_lat_v_abs_max,recovery_cutter_speed_mean,recovery_cutter_acc_mean,recovery_follower_speed_mean,recovery_follower_acc_mean,recovery_cutter_dy
0,1.0,9.0,17.0,1.0,2.0,109.0,4.36,169.60,3.798432,20.483092,...,51.0,169.60,3.798432,20.483092,0.52,36.491129,0.119216,44.760800,0.090196,0.49
1,1.0,11.0,12.0,2.0,1.0,164.0,6.56,60.64,1.771028,7.574775,...,22.0,60.64,1.771028,7.896714,0.49,26.591811,0.741818,34.515141,-0.673182,-0.23
2,1.0,23.0,22.0,2.0,1.0,628.0,25.12,-7.77,-0.317272,inf,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,30.0,32.0,1.0,2.0,588.0,23.52,105.86,3.222527,30.153453,...,50.0,105.90,3.222763,55.433673,0.36,31.716813,-0.007000,33.180038,0.395400,0.24
4,1.0,48.0,45.0,2.0,1.0,1148.0,45.92,-32.89,-1.414624,inf,...,50.0,30.23,1.305832,inf,0.41,32.219142,0.319200,23.130009,0.025800,-0.45


## SFC Decode Demo (3x3)

In [13]:
H = [[0, 1, 14, 15], [3, 2, 13, 12], [4, 7, 8, 11], [5, 6, 9, 10]]

def code_to_3x3(code: int):
    return [[(int(code) >> H[r][c]) & 1 for c in range(3)] for r in range(3)]

csv_path = step14_codes_csv_path()
if csv_path.exists():
    one = pd.read_csv(csv_path, usecols=['event_id', 'frame', 'code', 'code_hex'], nrows=1).iloc[0]
    matrix = code_to_3x3(int(one['code']))
    print('event_id:', int(one['event_id']), 'frame:', int(one['frame']), 'code:', int(one['code']), 'hex:', one['code_hex'])
    print('\n'.join(' '.join(str(v) for v in row) for row in matrix))
else:
    print('File not found:', csv_path)

event_id: 1 frame: 9 code: 16390 hex: 4006
0 1 1
0 1 0
0 0 0
